# MicroLakehouse (Iceberg + Spark + Trino + MinIO)

This notebook demonstrates how to create an Iceberg table using Spark and query it using Trino, all running locally in this Docker container.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MicroLakehouse") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session created successfully!")

In [ ]:
# Create a database and table in Iceberg (lakehouse catalog)
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.default")
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.default.taxis (
    vendor_id bigint,
    trip_id bigint,
    trip_distance float,
    fare_amount double,
    store_and_fwd_flag string
)
USING iceberg
PARTITIONED BY (vendor_id)
""")

print("Table created.")

In [ ]:
# Insert some sample data
spark.sql("""
INSERT INTO lakehouse.default.taxis
VALUES 
    (1, 1000371, 1.8, 15.32, 'N'),
    (2, 1000372, 2.5, 22.15, 'N'),
    (2, 1000373, 0.9, 9.01, 'Y'),
    (1, 1000374, 8.4, 42.13, 'Y')
""")

print("Data inserted.")

In [ ]:
# Read back using Spark
spark.sql("SELECT * FROM lakehouse.default.taxis").show()

## Querying with Trino

Now let's use Trino to query the exact same data. Trino is running on port 8080.

In [ ]:
!/opt/trino/bin/trino --server localhost:8080 --catalog lakehouse --schema default --execute "SELECT * FROM taxis"